In [64]:
import torch
import pandas as pd
from torch import nn
from torch.utils import data
import numpy as np

In [65]:
TRAIN_DATA_PATH = '../data/train.csv'
TEST_DATA_PATH = '../data/test.csv'
train_data = pd.read_csv(TRAIN_DATA_PATH)
test_data = pd.read_csv(TEST_DATA_PATH)

In [66]:
train_data.shape, test_data.shape, train_data.iloc[:5, [0, 1, 2, -3, -2, -1]], test_data.iloc[:5, [0, 1, 2, -3, -2, -1]]

((1460, 81),
 (1459, 80),
    Id  MSSubClass MSZoning SaleType SaleCondition  SalePrice
 0   1          60       RL       WD        Normal     208500
 1   2          20       RL       WD        Normal     181500
 2   3          60       RL       WD        Normal     223500
 3   4          70       RL       WD       Abnorml     140000
 4   5          60       RL       WD        Normal     250000,
      Id  MSSubClass MSZoning  YrSold SaleType SaleCondition
 0  1461          20       RH    2010       WD        Normal
 1  1462          20       RL    2010       WD        Normal
 2  1463          60       RL    2010       WD        Normal
 3  1464          60       RL    2010       WD        Normal
 4  1465         120       RL    2010       WD        Normal)

In [67]:
# 对train去掉Id和SalePrice，对test去掉Id
all_data = pd.concat([train_data.iloc[:, 1: -1], test_data.iloc[:, 1:]], axis = 0)

In [68]:
all_data.shape, all_data.iloc[1459:1462, [0, 1, 2, -3, -2, -1]]

((2919, 79),
       MSSubClass MSZoning  LotFrontage  YrSold SaleType SaleCondition
 1459          20       RL         75.0    2008       WD        Normal
 0             20       RH         80.0    2010       WD        Normal
 1             20       RL         81.0    2010       WD        Normal)

In [69]:
train_data.dtypes[(train_data.isnull().sum() > 0)][:5], test_data.dtypes[(test_data.isnull().sum() > 0)][:5]

(LotFrontage    float64
 Alley           object
 MasVnrType      object
 MasVnrArea     float64
 BsmtQual        object
 dtype: object,
 MSZoning        object
 LotFrontage    float64
 Alley           object
 Utilities       object
 Exterior1st     object
 dtype: object)

In [70]:
n_train = train_data.shape[0]
train_labels = torch.tensor(train_data.SalePrice.values.reshape(-1, 1), dtype = torch.float32)
train_data = all_data.iloc[:n_train, :]
test_data = all_data.iloc[n_train:, :]
train_labels.shape, train_labels[0]

(torch.Size([1460, 1]), tensor([208500.]))

In [71]:
train_data.shape, test_data.shape

((1460, 79), (1459, 79))

In [72]:
# Z-score 平均值0填充na
def pre_data(data):
    numeric_features = data.dtypes[data.dtypes != 'object'].index
    data.loc[:, numeric_features] = data.loc[:, numeric_features].apply(lambda x: (x-x.mean())/(x.std()))
    data.loc[:, numeric_features] = data.loc[:, numeric_features].fillna(0)
pre_data(train_data)
pre_data(test_data)

In [73]:
all_data = pd.concat([train_data, test_data], axis = 0)
all_data.shape

(2919, 79)

In [74]:
all_data = pd.get_dummies(all_data, dummy_na = True)
all_data.shape

(2919, 330)

In [75]:
train_data = all_data.iloc[:n_train, :]
test_data = all_data.iloc[n_train:, :]
train_data.shape, test_data.shape

((1460, 330), (1459, 330))

In [76]:
train_data.dtypes[train_data.dtypes == 'object'].index

Index([], dtype='object')

In [77]:
# 生成torch张量
train_data = train_data.astype('float32')
test_data = test_data.astype('float32')
train_data = torch.tensor(train_data.values, dtype = torch.float32)
test_data = torch.tensor(test_data.values, dtype = torch.float32)
train_data.shape, test_data.shape

(torch.Size([1460, 330]), torch.Size([1459, 330]))

In [78]:
train_dataset = data.TensorDataset(train_data, train_labels)
batch_size = 64
train_iter = data.DataLoader(train_dataset, batch_size = batch_size)

In [79]:
# 损失函数
loss = nn.MSELoss()

In [80]:
# 加载模型
in_features = train_data.shape[1]
out_features = 1
n_hiddens = 256
net = nn.Sequential(nn.Linear(in_features, n_hiddens), nn.ReLU(), nn.Linear(n_hiddens, out_features))

In [81]:
# 更新器
wd = 0
lr = 0.05
updater = torch.optim.Adam(net.parameters(), lr = lr, weight_decay= wd)

In [82]:
# 均方根误差
def log_rmse(y_features, y):
    y_features = torch.clamp(y_features, 1, float('inf'))
    return torch.square(loss(torch.log(y_features), torch.log(y)))

In [83]:
def train(train_iter, num_epochs, loss, net, updater):
    train_l = []

    for epoch in range(num_epochs):
        for X, y in train_iter:
            l = loss(net(X), y)
            updater.zero_grad()
            l.backward()
            updater.step()
        l_temp = log_rmse(net(train_data), train_labels)
        train_l.append(l_temp)
        print(f'epoch {epoch}, loss {l_temp}')
num_epochs = 110
train(train_iter, num_epochs, loss, net, updater)

epoch 0, loss 76.65766143798828
epoch 1, loss 2.54095721244812
epoch 2, loss 0.015053537674248219
epoch 3, loss 0.001987307332456112
epoch 4, loss 0.0009376714006066322
epoch 5, loss 0.0009424065356142819
epoch 6, loss 0.0007734103710390627
epoch 7, loss 0.00061272201128304
epoch 8, loss 0.0005099536501802504
epoch 9, loss 0.00044553098268806934
epoch 10, loss 0.00039985516923479736
epoch 11, loss 0.0003656450135167688
epoch 12, loss 0.0003384216397535056
epoch 13, loss 0.00031602391391061246
epoch 14, loss 0.0002973211812786758
epoch 15, loss 0.0002811115118674934
epoch 16, loss 0.00026691984385252
epoch 17, loss 0.0002543949522078037
epoch 18, loss 0.00024343760742340237
epoch 19, loss 0.0002338734338991344
epoch 20, loss 0.00022580321819987148
epoch 21, loss 0.00021879799896851182
epoch 22, loss 0.00021272744925227016
epoch 23, loss 0.00020744052017107606
epoch 24, loss 0.00020263245096430182
epoch 25, loss 0.00019834843988064677
epoch 26, loss 0.00019447212980594486
epoch 27, loss 

In [84]:
def pred_csv(net, test_data):
    y_hat = net(test_data).detach().numpy()
    submission = pd.DataFrame({
        'Id': torch.arange(1461, 2920).detach().numpy(),
        'SalePrice': y_hat.flatten()
    })
    submission.to_csv('../submission.csv', index = False)
pred_csv(net, test_data)